 ### Zoteroize and Obsidianize a Perplexity Dialogue



 In a Perplexity dialogue copied to the clipboard by the perplexity copy button and then saved to a file, replace

 the citation numbers with matching Obsidian literature note or Zotero item links

In [1]:
import re
import pathlib as pl
import sys
from collections import defaultdict
import pandas as pd
from icecream import ic

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as lpz

%load_ext autoreload
%autoreload 2


In [2]:
def split_body_source(perplexity_file: pl.Path):
    """Replace links in standard Perplexity (saved clipboard) output with links 
    to Zotero items or Obsidian lit notes. """    
    
    content = perplexity_file.read_text(encoding='utf-8')
    section_parts = content.split("\nCitations:\n", 1)
    if len(section_parts) < 2:
        print("Missing citations")
        body, citations = section_parts, ""
    else:
        body, citations = section_parts
    
    source_matches = list(re.finditer(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', citations, flags=re.M))

    url_to_source_num = {m.group('url'): m.group('num') for m in source_matches}
    
    return body, url_to_source_num, source_matches

def relink_chunks(body, url_to_source_num, source_matches) -> None:

    def make_relinks_from_source(cite_num: str, doc_url: str) -> str:
        """Returns what a relinked citation would look like if present in the body,
        given a source part citation number and url.  Also appends to the global list, 
        relinked_sources, a relinked source part link.  Expects the global set, body_cite_nums."""
        
        numbered_link = f"[{cite_num}]({doc_url})"
        if zotero_item := relinker.find_zotero_item_via_url(doc_url):
            body_link = relinker.create_obsidian_or_zotero_link(zotero_item)
            relinked_sources.append(f'({numbered_link}) **{body_link}**')
        else:
            body_link = f"=={numbered_link}==" # mark it as "not in zotero"
            source_line = f'({numbered_link}) {doc_url}'
            source_line = f'=={source_line} ==' if cite_num in body_cite_nums else source_line
            relinked_sources.append(source_line)
            
        return body_link

    relinker = lpz.ZoteroLinkConverter()
    relinked_sources = []
    body_cite_nums = set(re.findall(r'\[(\d+)\]', body))
    source_num_to_link = {m.group('num'): make_relinks_from_source(m.group('num'), m.group('url'))
                          for m in source_matches }
    
    body_relinked = re.sub(r'\[(\d+)\]', 
                           lambda m: f' {source_num_to_link.get(m.group(1))}', body)
    sources_relinked = "\n".join(relinked_sources)
    
    return body_relinked, sources_relinked

def relink_perplexity_export(perplexity_file: pl.Path, relinked_file: pl.Path) -> None:
    body, url_to_source_num, source_matches = split_body_source(perplexity_file)
    body_relinked, sources_relinked = relink_chunks(body, url_to_source_num, source_matches)
    relinked_file.write_text(f'# Response\n{body_relinked}\n# Citations\n{sources_relinked}', encoding='utf-8')
    


In [3]:
perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
output_file = rfw.refwrangle_test_dir / 'tmp' / "tmp_new_cites_perplexity_example.md"
print(f'{perplexity_dialog_file=}\n-->\n{output_file=}')

relink_perplexity_export(perplexity_dialog_file, output_file)
print('Done.')


perplexity_dialog_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/perplexity_example.md')
-->
output_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_new_cites_perplexity_example.md')
Reading from cache.
Done.


### Test merging

In [4]:
tmpdir = rfw.refwrangle_test_dir / 'tmp'
tmpdir.mkdir(parents=True, exist_ok=True)

datdir = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex'
datdir.mkdir(parents=True, exist_ok=True)

chat_files = list(datdir.glob('*.md'))
chat_files

merged_output_file = tmpdir / 'tmp_stock_perplexy_merged.md'

In [5]:

# get the sources from all docs to be merged
all_bodies, all_url_to_source_nums, all_source_matches = [], [], []
for chat_file in chat_files:
    body, url_to_source_num, source_matches = split_body_source(chat_file)
    all_bodies.append(body)
    all_source_matches.append(source_matches)
    all_url_to_source_nums.append(url_to_source_num)

# find all the doc citations for each unique URL
allurls = defaultdict(list)
print(allurls)
for docIx, url_to_source_num in enumerate(all_url_to_source_nums):
    for url, num in url_to_source_num.items():
        allurls[url].append(dict(orig_num=num, docIx=docIx))

# create new citenums for a combined document with a combined sources section
new_cite_num = 1
lut = []
for url, infos in allurls.items():
    for info in infos:
        lut.append({'url': url, 'new_cite_num': str(new_cite_num)} | info)
    new_cite_num += 1

lut = pd.DataFrame(lut).set_index(['docIx', 'orig_num'])
all_new_cite_nums = lut.new_cite_num.unique()


# make single body with cite numbers replaced by combined cite numbers

body_cites_not_in_sources = []
def replace_link_num(m):
    orig_citenum = m.group('orig')
    try:
        num = old_to_new_citenum[orig_citenum]
    except:
        chat_file=chat_files[docIx]
        print(f"Missing source for {orig_citenum} in {chat_file}")
        info = dict(orig_cite_num=orig_citenum, chat_file=chat_file)
        body_cites_not_in_sources.append(info)
        num = orig_citenum

    return f"[{num}]({m.group('url')})"

body_num_links_re = re.compile(r'\[(?P<orig>\d+)\]\((?P<url>https?://[^\)]+)\)')
sources_num_links_re = re.compile(r'\((?P<orig>\d+)\)\((?P<url>https?://[^\)]+)\)')

concat_bodies = ""
concat_sources = ""
for docIx, body in enumerate(all_bodies):
    # relink body with old citenums
    source_matches = all_source_matches[docIx]
    url_to_source_nums = all_url_to_source_nums[docIx]
    body_relinked, sources_relinked = relink_chunks(body, url_to_source_num, source_matches)
    
    old_to_new_citenum =lut.loc[docIx].new_cite_num.to_dict()
    body_re_relinked = re.sub(body_num_links_re, replace_link_num, body_relinked)
    concat_bodies += f'# {chat_files[docIx].name}\n{body_re_relinked}\n'
    sources_re_relinked = re.sub(body_num_links_re, replace_link_num, sources_relinked)
    concat_sources += f'{sources_re_relinked}\n'

defaultdict(<class 'list'>, {})
Reading from cache.
Reading from cache.
Reading from cache.
Reading from cache.
Reading from cache.
Missing source for 10 in C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\merge_chats_perplex\o3mini.md
Missing source for 11 in C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\merge_chats_perplex\o3mini.md
Missing source for 6 in C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\merge_chats_perplex\o3mini.md
Missing source for 7 in C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\merge_chats_perplex\o3mini.md
Missing source for 9 in C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\merge_chats_perplex\o3mini.md
Missing source for 10 in C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\merge_chats_perplex\o3mini.md
Missing source for 11 in C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\merge_chats_perplex\o3mini.md
Missing source for 12 in C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\merge_chats_perplex\o3mini.md

In [8]:
import numpy as np
len(concat_sources.split('\n')), len(np.unique(concat_sources.split('\n')))

concat_sources_unique = list(set(concat_sources.split('\n')))
#sorted_strings = sorted(concat_sources_unique, key=lambda x: int(re.search(r'\((\d+)\]', x).group(1)))
#sorted_strings

# Function to extract the number inside [num]
def extract_number(s):
    match = re.search(r'\[(\d+)\]', s)
    return int(match.group(1)) if match else float('inf')  # Handle cases without [num]

# Sort the list using the extracted number as key
merged_sources = "\n".join(sorted(concat_sources_unique, key=extract_number))

ic(merged_output_file)
merged_output_file.write_text(f'# Responses\n{concat_bodies}\n# Citations\n{merged_sources}', encoding='utf-8')
print('Done.')

ic| merged_output_file: WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_stock_perplexy_merged.md')


Done.


In [7]:
#pattern = r'\[(\d+)\]\(https?://[^\)]+\)'

#matches = re.findall(pattern, body_relinked)

# matches = list(re.finditer(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', body_relinked, flags=re.M))

# matches
#url_to_source_num = {m.group('url'): m.group('num') for m in matches}

pattern = r'\[(\d+)\]\(https?://[^\)]+\)'
old_to_new_citenum = lut.loc[docIx].new_cite_num.to_dict()
def replace_link_num(match):
    old_number = match.group(1)  # Extract the matched number
    return f"[{old_to_new_citenum.get(old_number, old_number)}]{match.group(0)[len(old_number)+2:]}"  # Replace if in dict

updated_text = re.sub(pattern, replace_link_num, text)

    


# body_relinked = re.sub(r'\[(\d+)\](
    # )', 
#                            lambda m: f' {source_num_to_link.get(m.group(1))}', body)

#matches

NameError: name 'text' is not defined

In [ ]:
import re

# Dictionary mapping old citation numbers to new ones
old_to_new_citenum = {'6': '2', '3': '1'}

# Test string
text = """
[6](https://www.groupcaliber.com/brand-tracking-101-what-every-business-needs-to-know-for-long-term-success/)
[3]
"""

# Regular expression with named groups for citation number (old) and URL (url)
pattern = r'\[(?P<old>\d+)\]\((?P<url>https?://[^\)]+)\)'


print(updated_text)


In [ ]:
url_to_source_num = lut.set_index('url')['new_cite_num'].to_dict()


relink_chunks(concat_bodies, url_to_source_num, output_file)



In [ ]:
print(body)

#body_relinked = re.sub(r'\[(\d+)\]', 
#                           lambda m: f' {source_num_to_link.get(m.group(1))}', body)



In [ ]:
#output_file

re.findall(r'\[(\d+)\]', lambda m: f' {source_num_to_link.get(m.group(1))}', body)


In [ ]:
display(lut)
#lut.loc[0, '3'].new_cite_num
all_new_cite_nums


In [ ]:
import re

# Input string and regex pattern
text = "apple banana apple orange banana"
pattern = r'\b(\w+)\b'  # Matches words

# Step 1: Extract all matches
matches = re.findall(pattern, text)

# Step 2: Compute unique substitutes
unique_substitutes = {match: f"word_{i}" for i, match in enumerate(set(matches), start=1)}

# Step 3: Define replacement function
def replacement_function(match):
    return unique_substitutes[match.group(0)]

# Step 4: Perform substitutions
result = re.sub(pattern, replacement_function, text)

print("Original:", text)
print("Modified:", result)



In [ ]:
# def relink_perplexity_export(perplexity_file: pl.Path, relinked_file: pl.Path) -> None:
#     """Replace links in standard Perplexity (saved clipboard) output with links 
#     to Zotero items or Obsidian lit notes."""    
    
#     relinker = lpz.ZoteroLinkConverter()

#     def make_relinks_from_source(cite_num: str, doc_url: str) -> str:
#         """Returns what a relinked citation would look like if present in the body,
#         given a source part citation number and url.  Also appends to the global list, 
#         relinked_sources, a relinked source part link.  Expects the global set, body_cite_nums."""
        
#         numbered_link = f"[{cite_num}]({doc_url})"
#         if zotero_item := relinker.find_zotero_item_via_url(doc_url):
#             body_link = relinker.create_obsidian_or_zotero_link(zotero_item)
#             relinked_sources.append(f'({numbered_link}) **{body_link}**')
#         else:
#             body_link = f"=={numbered_link}==" # mark it as "not in zotero"
#             source_line = f'({numbered_link}) {doc_url}'
#             source_line = f'=={source_line} ==' if cite_num in body_cite_nums else source_line
#             relinked_sources.append(source_line)
            
#         return body_link
    
#     content = perplexity_file.read_text(encoding='utf-8')
#     section_parts = content.split("\nCitations:\n", 1)
#     if len(section_parts) < 2:
#         print("Missing citations")
#         body, citations = section_parts, ""
#     else:
#         body, citations = section_parts
    
#     source_matches = re.finditer(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', citations, flags=re.M)

#     relinked_sources = []
#     body_cite_nums = set(re.findall(r'\[(\d+)\]', body))
#     url_to_source_num = {m.group('url'): m.group('num') for m in source_matches}
#     #ic(len(url_to_source_num))
#     source_num_to_link = {url: make_relinks_from_source(num, url) for url, num in url_to_source_num.items()}
#     #ic(len(source_num_to_link), len(url_to_source_num))
#     # source_num_to_link = {m.group('num'): make_relinks_from_source(m.group('num'), m.group('url'))
#     #                       for m in source_matches }
    
#     body_relinked = re.sub(r'\[(\d+)\]', 
#                            lambda m: f' {source_num_to_link.get(m.group(1))}', body)
#     sources_relinked = "\n".join(relinked_sources)
    
#     relinked_text = f"{body_relinked}\nCitations:\n{sources_relinked}"
#     relinked_file.write_text(f"{body_relinked}\nCitations:\n{sources_relinked}", 
#                              encoding='utf-8')
    
#     return relinked_text, source_num_to_link


